# Phase 3 — Fixed clean/weighted BiLSTM comparison and finalization

Runs only after the analysis notebook passes its fixed prerequisites. Train clean and weighted variants from the same initialization, with AdamW and the existing fixed settings. Select by validation MAE, write an immutable selection record, then evaluate only that selected model and four fixed baselines on holdout. Cache completed training/evaluation and verify hashes on reuse. The holdout is event/dependency-disjoint within this experiment, but belongs to the old model's former training population. No old-model evaluation on it and no cross-population improvement claim.


In [1]:
from pathlib import Path
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
import json,hashlib,time,random,platform,math
import numpy as np
import torch
# Narrow allowance for this trusted local checkpoint metadata type; weights_only remains enabled.
torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])
from torch import nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if ROOT.name!="bilstm new updated":ROOT=ROOT/"bilstm new updated"
ROOT=ROOT.resolve();OUT=ROOT/"phase3_bilstm_results";NEW=OUT/"modern_dataset_v1"
def readj(p):return json.loads(p.read_text(encoding="utf-8"))
def sha(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
def clean(v):
    if isinstance(v,dict):return {str(k):clean(x) for k,x in v.items()}
    if isinstance(v,(tuple,list,np.ndarray)):return [clean(x) for x in v]
    if isinstance(v,np.generic):return clean(v.item())
    if isinstance(v,float) and not np.isfinite(v):return None
    return v
def savej(p,v):
    temp=p.with_suffix(p.suffix+".tmp")
    temp.write_text(json.dumps(clean(v),indent=2,allow_nan=False),encoding="utf-8");temp.replace(p)
def savept(p,v):
    temp=p.with_suffix(".tmp");torch.save(v,temp);temp.replace(p)
protocol=readj(OUT/"protocol.json");decision=readj(OUT/"feasibility.json")
complete=readj(OUT/"analysis_complete.json")
assert complete["protocol_hash"]==decision["protocol_hash"]
ALLOWED=decision["training_allowed"]
CFG=protocol["model"];SEED=42
if ALLOWED:
    validity=readj(NEW/"validation.json")
    assert validity["status"]=="PASS" and sha(NEW/"modern_catalog_seq20.npz")==validity["dataset_sha256"]
    assert all(sha(NEW/name)==digest for name,digest in complete["artifacts"].items())
    with np.load(NEW/"modern_catalog_seq20.npz",allow_pickle=False) as z:
        train_X=torch.from_numpy(z["X_train"].copy());train_y=torch.from_numpy(z["y_train"].copy())
        val_X=torch.from_numpy(z["X_val"].copy());val_y=z["y_val"].astype(np.float64)
        val5=z["mean5_val"].copy();val20=z["mean20_val"].copy()
    torch.set_num_threads(4);torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_gpu=train_X.to(DEVICE);val_gpu=val_X.to(DEVICE)
    weights=1+.5*np.clip((train_y.numpy().astype(float)-4)/1.5,0,1)
    notebook=readj(ROOT/"phase3_modern_bilstm_training.ipynb")
    source="\n".join("".join(c["source"]) for c in notebook["cells"] if c["cell_type"]=="code")
    run={"protocol_hash":complete["protocol_hash"],"dataset_sha256":validity["dataset_sha256"],
         "scaler_sha256":validity["scaler_sha256"],"training_source_sha256":hashlib.sha256(source.encode()).hexdigest(),
         "settings":CFG,"seed":42,"torch":torch.__version__,"numpy":np.__version__,"python":platform.python_version(),
         "device":str(DEVICE),"gpu":torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else None,
         "cuda":torch.version.cuda,"cudnn":torch.backends.cudnn.version(),"deterministic":True,"TF32":False,
         "sampler":protocol["sampler"],"sampling_weights_sha256":hashlib.sha256(weights.tobytes()).hexdigest()}
    run_hash=hashlib.sha256(json.dumps(run,sort_keys=True).encode()).hexdigest()
    if (OUT/"run_configuration.json").exists():
        recorded=readj(OUT/"run_configuration.json")
        if recorded!=run:
            # Only this documented metadata compatibility amendment can reuse the initial run identity.
            original_nb=readj(OUT/"provenance/initial_training_notebook.ipynb")
            original_source="\n".join("".join(c["source"]) for c in original_nb["cells"] if c["cell_type"]=="code")
            assert hashlib.sha256(original_source.encode()).hexdigest()==recorded["training_source_sha256"]
            assert "".join(notebook["cells"][2]["source"])=="".join(original_nb["cells"][2]["source"]),"Model/optimizer/training changed"
            assert "".join(notebook["cells"][3]["source"])=="".join(original_nb["cells"][3]["source"]),"Evaluation plan changed"
            assert {k:v for k,v in recorded.items() if k!="training_source_sha256"}=={k:v for k,v in run.items() if k!="training_source_sha256"}
            amendment={"initial_source_sha256":recorded["training_source_sha256"],"executed_source_sha256":run["training_source_sha256"],
                "change":"Allow torch.torch_version.TorchVersion metadata in weights_only loader; model/optimizer/evaluation cells byte-identical",
                "resume":"Clean stopping state resumed without repeated optimization; weighted starts with same fixed settings",
                "initial_run_configuration_retained":True}
            if (OUT/"source_amendment.json").exists():assert readj(OUT/"source_amendment.json")==amendment
            else:savej(OUT/"source_amendment.json",amendment)
            run=recorded
            run_hash=hashlib.sha256(json.dumps(run,sort_keys=True).encode()).hexdigest()
    else:savej(OUT/"run_configuration.json",run)
    print("Fixed experiment",str(DEVICE),"; train/validation:",len(train_y),len(val_y))
else:print("Training skipped:",decision["reason"])


Fixed experiment cuda ; train/validation: 692 346


In [2]:
def seed_all():
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
class BiLSTMRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(10,CFG["hidden_size"],num_layers=CFG["layers"],
                          dropout=CFG["dropout"],batch_first=True,bidirectional=True)
        self.dropout=nn.Dropout(CFG["dropout"]);self.head=nn.Linear(2*CFG["hidden_size"],1)
    def forward(self,x):
        _,(h,_)=self.lstm(x)
        return self.head(self.dropout(torch.cat([h[-2],h[-1]],dim=1))).squeeze(-1)
def state_hash(model):
    h=hashlib.sha256()
    for name,tensor in model.state_dict().items():
        h.update(name.encode());h.update(tensor.detach().cpu().numpy().tobytes())
    return h.hexdigest()
def predict(model,x):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(x[i:i+256]).cpu().numpy() for i in range(0,len(x),256)]).astype(float)
def metrics(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float);e=p-y;hi=y>=4.5
    return dict(n=len(y),mae=float(np.abs(e).mean()),rmse=float(np.sqrt((e**2).mean())),
        r2=float(1-(e**2).sum()/((y-y.mean())**2).sum()),pred_mean=float(p.mean()),pred_sd=float(p.std()),
        actual_mean=float(y.mean()),actual_sd=float(y.std()),pred_actual_sd_ratio=float(p.std()/y.std()),
        high_n=int(hi.sum()),high_mae=float(np.abs(e[hi]).mean()) if hi.any() else None,
        high_underprediction_rate=float((e[hi]<0).mean()) if hi.any() else None,
        high_bias=float(e[hi].mean()) if hi.any() else None)
def rng_state():
    return {"python":random.getstate(),"torch":torch.get_rng_state(),
            "cuda":torch.cuda.get_rng_state_all() if DEVICE.type=="cuda" else []}
def restore_rng(s):
    random.setstate(s["python"]);torch.set_rng_state(s["torch"])
    if DEVICE.type=="cuda":torch.cuda.set_rng_state_all(s["cuda"])
def fit_variant(variant):
    folder=OUT/"models"/variant;folder.mkdir(parents=True,exist_ok=True)
    summary_path=folder/"fit_summary.json"
    if summary_path.exists():
        saved=readj(summary_path)
        assert saved["run_hash"]==run_hash and sha(folder/"best.pt")==saved["checkpoint_sha256"]
        return saved
    seed_all();model=BiLSTMRegressor().to(DEVICE);initial_hash=state_hash(model)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG["learning_rate"],weight_decay=CFG["weight_decay"])
    loss_fn=nn.HuberLoss(delta=1.0)
    history=[];best=float("inf");best_epoch=0;stale=0;elapsed=0.;start=1
    if (folder/"last.pt").exists():
        last=torch.load(folder/"last.pt",map_location="cpu",weights_only=True)
        assert last["run_hash"]==run_hash
        model.load_state_dict(last["model"]);opt.load_state_dict(last["optimizer"])
        history=last["history"];best=last["best"];best_epoch=last["best_epoch"]
        stale=last["stale"];elapsed=last["elapsed"];start=last["epoch"]+1
        restore_rng(last["rng"])
    for epoch in range(start,CFG["max_epochs"]+1):
        if stale>=CFG["patience"]:break
        began=time.perf_counter()
        if variant=="clean":parents=np.arange(len(train_y))
        else:
            rng=np.random.default_rng(np.random.SeedSequence([SEED,epoch-1]))
            parents=np.sort(rng.choice(len(train_y),len(train_y),replace=True,p=weights/weights.sum()))
        model.train();loss_sum=0.
        for batch in range(0,len(parents),CFG["batch_size"]):
            ids=torch.as_tensor(parents[batch:batch+CFG["batch_size"]],device=DEVICE)
            target=train_y[parents[batch:batch+CFG["batch_size"]]].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=loss_fn(model(train_gpu[ids]),target)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
            loss_sum+=float(loss.detach())*len(ids)
        tp=predict(model,train_gpu);vp=predict(model,val_gpu)
        tm=metrics(train_y.numpy(),tp);vm=metrics(val_y,vp)
        improved=vm["mae"]<best
        if improved:
            best=vm["mae"];best_epoch=epoch;stale=0
            savept(folder/"best.pt",{"model_state":{k:v.detach().cpu() for k,v in model.state_dict().items()},
                "run_hash":run_hash,"configuration":run,"variant":variant,"best_epoch":epoch,"validation_mae":best,
                "initialization_sha256":initial_hash,"selection":"validation MAE only"})
        else:stale+=1
        elapsed+=time.perf_counter()-began
        history.append({"epoch":epoch,"train_huber":loss_sum/len(parents),"train":tm,"val":vm,
                        "parent_indices_sha256":hashlib.sha256(parents.astype(np.int64).tobytes()).hexdigest(),
                        "unique_parents":len(np.unique(parents)),"draw_high_fraction":float((train_y.numpy()[parents]>=4.5).mean()),
                        "elapsed_seconds":elapsed})
        savej(folder/"history.json",history)
        savept(folder/"last.pt",{"run_hash":run_hash,"model":model.state_dict(),"optimizer":opt.state_dict(),
            "epoch":epoch,"history":history,"best":best,"best_epoch":best_epoch,"stale":stale,"elapsed":elapsed,"rng":rng_state()})
    best_ck=torch.load(folder/"best.pt",map_location="cpu",weights_only=True)
    model.load_state_dict(best_ck["model_state"])
    tp=predict(model,train_gpu);vp=predict(model,val_gpu)
    np.savez_compressed(folder/"validation_predictions.npz",actual=val_y,prediction=vp)
    np.savez_compressed(folder/"training_predictions.npz",actual=train_y.numpy(),prediction=tp)
    summary={"run_hash":run_hash,"variant":variant,"best_epoch":best_epoch,"epochs_run":len(history),
             "training_seconds":elapsed,"train":metrics(train_y.numpy(),tp),"val":metrics(val_y,vp),
             "initialization_sha256":initial_hash,"parameter_count":sum(p.numel() for p in model.parameters()),
             "checkpoint_sha256":sha(folder/"best.pt")}
    savej(summary_path,summary)
    print(variant,"best epoch",best_epoch,"validation MAE",round(summary["val"]["mae"],6),"epochs",len(history))
    return summary
if ALLOWED:
    summaries={variant:fit_variant(variant) for variant in ["clean","weighted"]}
    assert summaries["clean"]["initialization_sha256"]==summaries["weighted"]["initialization_sha256"]
    winner=min(summaries,key=lambda v:(summaries[v]["val"]["mae"],v))
    selection={"run_hash":run_hash,"selected_variant":winner,
               "selected_checkpoint_sha256":summaries[winner]["checkpoint_sha256"],
               "selection_metric":"validation MAE","validation_candidates":{v:r["val"]["mae"] for v,r in summaries.items()},
               "dataset_sha256":validity["dataset_sha256"],"holdout_metrics_accessed_for_selection":False}
    if (OUT/"selection_lock.json").exists():assert readj(OUT/"selection_lock.json")==selection
    else:savej(OUT/"selection_lock.json",selection)
    validation_metrics={v:s["val"] for v,s in summaries.items()}
    baseline_val={"train_mean":np.full(len(val_y),train_y.numpy().astype(float).mean()),
                  "train_median":np.full(len(val_y),np.median(train_y.numpy().astype(float))),
                  "recent5_mean":val5,"recent20_mean":val20}
    validation_metrics.update({k:metrics(val_y,p) for k,p in baseline_val.items()})
    savej(OUT/"validation_metrics.json",validation_metrics)
    print("Selection frozen:",winner,"; holdout not yet evaluated.")


Selection frozen: clean ; holdout not yet evaluated.


In [3]:
if ALLOWED:
    # Freeze benchmark and reporting rule before accessing holdout arrays.
    best_baseline=min(baseline_val,key=lambda k:(validation_metrics[k]["mae"],k))
    evaluation_plan={"run_hash":run_hash,"selection":selection,"baseline_selected_on_validation":best_baseline,
        "holdout_model_evaluations":"Only selected BiLSTM; losing model never evaluated on holdout",
        "bootstrap":{"seed":42,"replicates":1000,"block":20,"method":"paired moving-block percentile, conditional on frozen models"},
        "adequacy_rule":"Retain modern as research candidate only if val MAE beats best validation baseline, holdout R2>0, paired holdout MAE-difference CI upper<0, and high MAE no worse than that baseline. Otherwise C: neither established as adequate; keep research baselines.",
        "uncertainty":"One seed and retrospective catalog reuse; intervals conditional, not deployment guarantees"}
    plan_path=OUT/"evaluation_plan.json"
    if plan_path.exists():assert readj(plan_path)==evaluation_plan
    else:savej(plan_path,evaluation_plan)
    cache=OUT/"holdout_metrics.json";predcache=OUT/"holdout_predictions.npz"
    if cache.exists():
        evaluation=readj(cache)
        assert evaluation["run_hash"]==run_hash and evaluation["selection"]==selection
        assert sha(predcache)==evaluation["predictions_sha256"]
        with np.load(predcache,allow_pickle=False) as z:
            hold_y=z["actual"];hold_predictions={k:z[k] for k in z.files if k!="actual"}
        print("Reused locked holdout evaluation; no new holdout inference.")
    else:
        # Persist predictions before calculations, so interruption after inference can reuse them.
        attempt=OUT/"holdout_access.json"
        if predcache.exists():
            assert readj(attempt)["selection"]==selection
            with np.load(predcache,allow_pickle=False) as z:
                hold_y=z["actual"];hold_predictions={k:z[k] for k in z.files if k!="actual"}
        else:
            assert not attempt.exists(),"An interrupted inference attempt is recorded; do not silently repeat holdout access"
            savej(attempt,{"state":"STARTED","selection":selection,"run_hash":run_hash,
                           "selection_lock_sha256":sha(OUT/"selection_lock.json")})
            with np.load(NEW/"modern_catalog_seq20.npz",allow_pickle=False) as z:
                hold_X=torch.from_numpy(z["X_holdout"].copy()).to(DEVICE)
                hold_y=z["y_holdout"].astype(float)
                hold_predictions={"train_mean":np.full(len(hold_y),train_y.numpy().astype(float).mean()),
                    "train_median":np.full(len(hold_y),np.median(train_y.numpy().astype(float))),
                    "recent5_mean":z["mean5_holdout"].copy(),"recent20_mean":z["mean20_holdout"].copy()}
            ck=torch.load(OUT/"models"/winner/"best.pt",map_location="cpu",weights_only=True)
            assert ck["run_hash"]==run_hash
            model=BiLSTMRegressor().to(DEVICE);model.load_state_dict(ck["model_state"])
            hold_predictions["selected_"+winner]=predict(model,hold_X)
            np.savez_compressed(predcache,actual=hold_y,**hold_predictions)
            savej(attempt,{"state":"PREDICTIONS_SAVED","selection":selection,"run_hash":run_hash,
                           "predictions_sha256":sha(predcache),"selected_model_inference_passes":1})
            del model,hold_X
        hold_metrics={k:metrics(hold_y,p) for k,p in hold_predictions.items()}
        model_key="selected_"+winner;error=np.abs(hold_predictions[model_key]-hold_y)
        comparison=error-np.abs(hold_predictions[best_baseline]-hold_y)
        rng=np.random.default_rng(42);deltas=[];maes=[]
        for _ in range(1000):
            starts=rng.integers(0,len(hold_y)-20+1,size=math.ceil(len(hold_y)/20))
            ix=(starts[:,None]+np.arange(20)).ravel()[:len(hold_y)]
            deltas.append(comparison[ix].mean());maes.append(error[ix].mean())
        ci=np.quantile(deltas,[.025,.975])
        chosen_metrics=hold_metrics[model_key]
        adequate=(summaries[winner]["val"]["mae"]<validation_metrics[best_baseline]["mae"]
                  and chosen_metrics["r2"]>0 and ci[1]<0
                  and chosen_metrics["high_n"]>0
                  and chosen_metrics["high_mae"]<=hold_metrics[best_baseline]["high_mae"])
        evaluation={"run_hash":run_hash,"selection":selection,"metrics":hold_metrics,
            "predictions_sha256":sha(predcache),"model_mae_ci95_block20":np.quantile(maes,[.025,.975]),
            "paired_mae_difference_model_minus_baseline":float(comparison.mean()),
            "paired_mae_difference_ci95_block20":ci,"comparison_baseline":best_baseline,
            "final_verdict":"B" if adequate else "C",
            "final_interpretation":"Retain modern as limited retrospective research candidate" if adequate else "Neither established as adequately generalizing; preserve original and modern as research baselines"}
        savej(cache,evaluation)
    # Essential curves and one holdout diagnostic; no duplicate model evaluations.
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for variant,ax in zip(["clean","weighted"],axes):
        hh=readj(OUT/"models"/variant/"history.json")
        ax.plot([r["epoch"] for r in hh],[r["train"]["mae"] for r in hh],label="clean train MAE")
        ax.plot([r["epoch"] for r in hh],[r["val"]["mae"] for r in hh],label="validation MAE")
        ax.axvline(summaries[variant]["best_epoch"],ls=":",color="gray")
        ax.set_title(variant);ax.set_xlabel("Epoch");ax.set_ylabel("MAE (ML)");ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(OUT/"training_validation_curves.png",dpi=140);plt.close(fig)
    pp=hold_predictions["selected_"+winner]
    fig,ax=plt.subplots(figsize=(5,4));ax.scatter(hold_y,pp,s=10,alpha=.45)
    low=min(hold_y.min(),pp.min());high=max(hold_y.max(),pp.max())
    ax.plot([low,high],[low,high],"k--");ax.set_xlabel("Actual holdout ML");ax.set_ylabel("Predicted ML")
    ax.set_title("Frozen selected "+winner);fig.tight_layout();fig.savefig(OUT/"holdout_actual_predicted.png",dpi=140);plt.close(fig)
    error_ranges=[]
    for lower,upper,label in [(-np.inf,3,"<3"),(3,4,"3 to <4"),(4,4.5,"4 to <4.5"),(4.5,np.inf,">=4.5")]:
        take=(hold_y>=lower)&(hold_y<upper)
        error_ranges.append({"range":label,"n":int(take.sum()),
             "mae":float(np.abs(pp[take]-hold_y[take]).mean()) if take.any() else None,
             "bias":float((pp[take]-hold_y[take]).mean()) if take.any() else None})
    savej(OUT/"holdout_error_by_magnitude.json",error_ranges)
    print("Holdout:",json.dumps(clean(evaluation["metrics"]["selected_"+winner])))
    print("Final research verdict:",evaluation["final_verdict"],"; paired MAE-difference CI",evaluation["paired_mae_difference_ci95_block20"])


Reused locked holdout evaluation; no new holdout inference.
Holdout: {"n": 366, "mae": 0.47305197663645926, "rmse": 0.5894757500379793, "r2": -0.006127542886309234, "pred_mean": 3.252791038627833, "pred_sd": 0.0618526979046736, "actual_mean": 3.2751366115007245, "actual_sd": 0.5876779887572077, "pred_actual_sd_ratio": 0.10524930163792015, "high_n": 10, "high_mae": 1.5451816082000733, "high_underprediction_rate": 1.0, "high_bias": -1.5451816082000733}
Final research verdict: C ; paired MAE-difference CI [-0.019381833363751922, 0.008360270900804465]


In [4]:
# Research report and artifact checks; old numerical results remain untouched.
def mdtable(headers,rows):
    return "\n".join(["| "+" | ".join(headers)+" |","|"+"|".join(["---"]*len(headers))+"|"]+
                     ["| "+" | ".join(str(x) for x in row)+" |" for row in rows])
mc=readj(OUT/"completeness_analysis.json")["estimates"]
def show_interval(v):
    return "not estimable" if v is None else f'[{v[0]:.2f}, {v[1]:.2f}]'
mc_table=mdtable(["Training period","n","Bin width","MAXC","MAXC interval","GFT","GFT interval","GFT success"],[
    (r["period"],r["n"],r["width"],r["maxc"],show_interval(r.get("maxc_bootstrap_percentile95")),
     "not estimable" if r["gft"] is None else f'{r["gft"]:.2f}',
     show_interval(r.get("gft_bootstrap_percentile95")),
     f'{100*r.get("gft_bootstrap_success_fraction",0):.0f}%') for r in mc])
mc_report=f"""# Training-period completeness analysis

{mc_table}

Conditional catalog-based estimates only: original retained ML magnitudes, training-period records before2023, no threshold-based data filtering. MAXC is the modal bin; GFT selects the first95% fit threshold (90% fallback), requiring100 tail events. Bin centers are integer multiples of width; deterministic half-bin ties go upward. These are adaptations, not a claimed exact reproduction of the reference's search grid or bootstrap.

Intervals are95% percentile ranges from200 moving-block resamples of20 consecutive events. GFT intervals condition on successful fits; success fractions quantify failures. They do not include magnitude calibration, spatial heterogeneity or unknown reporting bias and are not formal detection-completeness confidence bounds. Small samples or failed fits are marked unavailable. Bin-width variation is a required sensitivity check; rounded ML heaping can move modes.

Method reference: [Mignan & Woessner (2012), sections6.1–6.2 and appendix](https://www.corssa.org/export/sites/corssa/.galleries/articles-pdf/Mignan-Woessner-2012-CORSSA-Magnitude-of-completeness.pdf_2063069299.pdf). MAXC can underestimate completeness for heterogeneous catalogs. No current2026 Mc estimate or causal network-completeness conclusion is made.
"""
(OUT/"completeness_report.md").write_text(mc_report,encoding="utf-8")
if ALLOWED:
    all_validation=readj(OUT/"validation_metrics.json")
    vtable=mdtable(["Validation model","MAE","RMSE","R2","High n","High MAE","High underprediction","Prediction SD"],[
        (name,f'{v["mae"]:.6f}',f'{v["rmse"]:.6f}',f'{v["r2"]:.6f}',v["high_n"],
         f'{v["high_mae"]:.6f}' if v["high_mae"] is not None else "NA",
         f'{100*v["high_underprediction_rate"]:.1f}%' if v["high_underprediction_rate"] is not None else "NA",
         f'{v["pred_sd"]:.6f}') for name,v in all_validation.items()])
    htable=mdtable(["Frozen holdout model","MAE","RMSE","R2","High n","High MAE","High underprediction","Prediction SD"],[
        (name,f'{v["mae"]:.6f}',f'{v["rmse"]:.6f}',f'{v["r2"]:.6f}',v["high_n"],
         f'{v["high_mae"]:.6f}' if v["high_mae"] is not None else "NA",
         f'{100*v["high_underprediction_rate"]:.1f}%' if v["high_underprediction_rate"] is not None else "NA",
         f'{v["pred_sd"]:.6f}') for name,v in evaluation["metrics"].items()])
    gaps=mdtable(["Variant","Best/total epochs","Clean train MAE","Validation-train MAE","Saved duration seconds"],[
        (name,f'{v["best_epoch"]}/{v["epochs_run"]}',f'{v["train"]["mae"]:.6f}',
         f'{v["val"]["mae"]-v["train"]["mae"]:.6f}',f'{v["training_seconds"]:.3f}') for name,v in summaries.items()])
    selected=evaluation["metrics"]["selected_"+winner]
    modern_mc=[r for r in mc if r["period"]=="2020_2022"]
    report=f"""# Final CSV-based BiLSTM research report — Phase 3

**Verdict {evaluation["final_verdict"]}: {evaluation["final_interpretation"]}.** Selected by validation only: **{winner}**. No further tuning or original-test evaluation was performed.

## Design and provenance

Original dataset_2.csv hash matches verified provenance; retained ML eligibility/IDs and Magnitude_Value targets are unchanged. No old tensors, fitted scalers or weights were used to construct/train the modern experiment. The 10 historical feature definitions and sequence length20 remain fixed. Rolling features were recomputed in explicit time units. Train-only median imputation/standardization fits unique training-input events. Original artifacts remain unchanged.

The fixed newest-eligible training cutoff is2020, chosen only from pre-2023 targets using prespecified sample/representation and two-half mean/SD checks. It yields **692 training sequences,39 ML>=4.5**. These are exploratory sufficiency thresholds, not proof of stationarity or statistical power. Candidate details and all10 two-half feature correlations are saved in feasibility.json and training_feature_stability.csv; no validation/holdout performance selected the cutoff.

Validation is calendar2023 (**346 samples**); holdout is2024 through {protocol["holdout_through"]} (**366 samples**). Boundary samples were purged until all inputs, targets and rolling-feature dependency events stayed within their partition. Event/dependency sets are disjoint across splits; every input precedes its target. Purged counts differ from unpurged calendar counts.

**Holdout limitation:** this newly frozen partition previously belonged to the original model's training catalog. It is an internal retrospective experiment, not newly collected or historically unseen external data. Fixed architecture/hyperparameters also inherit prior research choices. The original validation/test populations are excluded, and the old model is not evaluated on its own former training events. Old test MAE0.5477/R2-0.0899 and these new holdout scores are not directly comparable proof of improvement.

## Completeness and feature stability

See [completeness_report.md](completeness_report.md) for all periods, bin widths0.1/0.2/0.3,200 block-bootstrap intervals, and GFT failures. Modern2020–2022 estimates:
{mdtable(["Bin width","MAXC","MAXC95% interval","GFT","GFT95% interval","GFT success"],[(r["width"],r["maxc"],show_interval(r.get("maxc_bootstrap_percentile95")),r["gft"],show_interval(r.get("gft_bootstrap_percentile95")),r.get("gft_bootstrap_success_fraction")) for r in modern_mc])}

These are conditional FMD estimates in a geographically pooled, retained-ML catalog, not established physical detection thresholds. Reporting completeness, source magnitude calibration and current2026 Mc remain unverified. No below-Mc labels were removed, no declustering was applied, and no causal completeness explanation is claimed. Training half means for the selected cutoff differ by {decision["candidates"][0]["half_mean_gap"]:.6f} ML; univariate relationships remain descriptive and do not certify forecasting information.

## Fixed comparison and validation

Both variants use fresh identical initialization: hidden128,2 layers, dropout {CFG["dropout"]}, AdamW learning rate {CFG["learning_rate"]}, weight decay {CFG["weight_decay"]}, batch128, Huber delta1, seed42, max50 epochs, patience8 and clipping1.0. Weighted training draws complete windows with replacement using weights1+0.5*clip((ML-4)/1.5,0,1), then sorts parent indices. Labels are unchanged. No noise or interpolation was added. Full runtime, source hashes, sampler weights, per-epoch parent hashes and RNG state are saved for reproducibility.

{vtable}

Selection was frozen in selection_lock.json before holdout array access. Baselines are training mean/median and historical last5/20-event magnitude means. Their formulas are fixed; the paired comparison baseline ({best_baseline}) is selected by validation MAE.

## One frozen holdout evaluation

{htable}

Only the selected BiLSTM was evaluated on holdout; the losing model remains unevaluated there. Saved predictions and an access record prevent routine notebook reruns from reevaluating holdout. Selected MAE95% moving-block interval: {show_interval(evaluation["model_mae_ci95_block20"])}. Paired MAE difference against {best_baseline}: {evaluation["paired_mae_difference_model_minus_baseline"]:.6f},95% block20 interval {show_interval(evaluation["paired_mae_difference_ci95_block20"])}. This is conditional one-seed uncertainty; chronology, model selection and prior catalog exposure limit generality.

Holdout actual SD={selected["actual_sd"]:.6f}; prediction SD={selected["pred_sd"]:.6f}; ratio={selected["pred_actual_sd_ratio"]:.4f}. High-magnitude group n={selected["high_n"]}, MAE={selected["high_mae"]}, underprediction={selected["high_underprediction_rate"]}. A small high group limits precision. Error by magnitude range is saved separately.

## Overfitting and final interpretation

{gaps}

[Training/validation curves](training_validation_curves.png) retain all epochs and best-epoch markers. Positive gaps and post-best validation deterioration can indicate overfitting, but chronology-induced distribution differences also contribute; gaps alone do not identify causes. Early stopping selects validation minima. Prediction compression is reported explicitly; greater variability alone would not prove improvement.

The prespecified adequacy screen requires better validation MAE than the best baseline, positive holdout R2, a paired MAE-difference interval wholly below zero and no worse high-magnitude MAE. It is a conservative research decision rule, not a universal standard. Result: **{evaluation["final_verdict"]}**. Preserve original Variant C and both modern fits as research artifacts; stop here.

Selected modern checkpoint: [models/{winner}/best.pt](models/{winner}/best.pt). Original research baseline remains ../../bilstm new updated/phase2_bilstm_results/variants/C/best.pt (unchanged). Dataset/scalers/manifests: modern_dataset_v1/. Training and evaluation code: ../phase3_modern_catalog_analysis.ipynb and ../phase3_modern_bilstm_training.ipynb.

This standalone catalog model is not independently validated seconds-ahead EEW. Historical publication times are unknown. Any future CNN fusion requires shared event IDs, causal cutoffs, compatible targets and common partitions; no CNN or fusion work was performed.
"""
    # Correct relative reference to original project-local checkpoint.
    report=report.replace("../../bilstm new updated/phase2_bilstm_results/variants/C/best.pt","../phase2_bilstm_results/variants/C/best.pt")
    final_decision={"verdict":evaluation["final_verdict"],"interpretation":evaluation["final_interpretation"],
                    "selected_modern_variant":winner,"checkpoint":str((OUT/"models"/winner/"best.pt").relative_to(ROOT)),
                    "dataset_validity":"PASS","original_test_reused":False,"no_operational_EEW_claim":True}
else:
    report=f"# Final BiLSTM research report\n\nVerdict A: retain original Variant C as an existing research baseline; modern experiment blocked by {decision['reason']}. No new training or holdout evaluation. Original weak generalization remains unresolved. See completeness_report.md and feasibility.json. No EEW or causal completeness claim."
    final_decision={"verdict":"A","reason":decision["reason"],"new_training":False}
(OUT/"final_bilstm_report.md").write_text(report,encoding="utf-8")
savej(OUT/"final_decision.json",final_decision)
protected=readj(ROOT/"phase2_bilstm_results/gemini_verification/correction_provenance_v1.json")["protected_hashes_after"]
assert all(sha(ROOT/rel)==digest for rel,digest in protected.items())
assert sha(ROOT.parent/"dataset_2.csv")==protocol["source_sha256"]
savej(OUT/"artifact_hashes.json",{"original_protected_sources_unchanged":True,
   "files":{str(p.relative_to(OUT)):sha(p) for p in OUT.rglob("*") if p.is_file() and p.name!="artifact_hashes.json"}})
marker="## Phase 3 CSV BiLSTM branch finalized — 2026-11-10"
note=f"""{marker}

- Final verdict {final_decision["verdict"]}: {final_decision.get("interpretation",final_decision.get("reason"))}.
- Fixed training-only period selection; no Optuna, CNN changes or original-test tuning. Phase 3 report: phase3_bilstm_results/final_bilstm_report.md.
- Split counts: {decision["split_counts"]}; all rolling-feature dependencies purged across boundaries; train-only new scaling.
- Holdout is an internal 2024 partition previously in original-model training, not unseen external validation. Publication availability and independently calibrated Mc remain unverified.
- Code: phase3_modern_catalog_analysis.ipynb and phase3_modern_bilstm_training.ipynb. Original artifacts preserved. No further tuning or fusion started.

"""
for p in [ROOT/"project_status.md",ROOT/".agent-memory/checkpoint.md"]:
    old=p.read_text(encoding="utf-8")
    if marker not in old:p.write_text(note+old,encoding="utf-8")
print("Final report saved; protected sources unchanged; final decision",final_decision["verdict"])


Final report saved; protected sources unchanged; final decision C
